# TRAM: SecBERT / RoBERTa retry (30 epochs, higher head learning rate)

Run the cells **top to bottom, one at a time**. Do not use "Run all" after editing.

Outputs go to `run_tram_2_epoc_30` (the old `runs` folder is never touched).

| Cell | What it does |
|---|---|
| 1 | Paths, imports, Drive mount |
| 2 | Load data + metric helpers |
| 3 | TF-IDF + SVM reference (seconds, CPU) |
| 4 | Defines `train_model` (nothing runs yet) |
| 5 | The 4 training runs (GPU: Runtime > Change runtime type > T4) |
| 6 | Results table |

In [1]:
# Cell 1 - setup, paths, imports
import os, ast, json, random, warnings
import numpy as np
import pandas as pd

try:
    from google.colab import drive
    drive.mount("/content/drive")
except ImportError:
    pass

DATA_DIR = "/content/drive/MyDrive/KDU/Thesis/Dataset"
OUT_ROOT = "/content/drive/MyDrive/KDU/Thesis/run_tram_2_epoc_30"   # NEW folder (old "runs" is untouched)
os.makedirs(OUT_ROOT, exist_ok=True)

MODELS = {"secbert": "jackaduma/SecBERT", "roberta": "roberta-base"}
warnings.filterwarnings("ignore", message="unknown class")
print("OUT_ROOT =", OUT_ROOT)

Mounted at /content/drive
OUT_ROOT = /content/drive/MyDrive/KDU/Thesis/run_tram_2_epoc_30


In [2]:
# Cell 2 - data + metric helpers (same label space and metrics as the old notebook)
from sklearn.preprocessing import MultiLabelBinarizer
from sklearn.metrics import f1_score, precision_score, recall_score, hamming_loss

def load_split(name):
    df = pd.read_csv(f"{DATA_DIR}/tram_{name}.tsv", sep="\t").rename(columns={"text1": "text"})
    df["y"] = df["labels"].map(lambda x: sorted(set(ast.literal_eval(x))))   # parse + de-duplicate labels
    return df[["text", "y"]]

train, dev, test = (load_split(s) for s in ("train", "dev", "test"))
train = train.drop_duplicates(subset=["text"]).reset_index(drop=True)       # train only; dev/test untouched
mlb = MultiLabelBinarizer().fit(train["y"])                                  # label space = labels seen in train
Y_train, Y_dev, Y_test = (mlb.transform(d["y"]) for d in (train, dev, test))
print(f"train={len(train)} dev={len(dev)} test={len(test)} classes={len(mlb.classes_)}")
assert len(train) == 3468, f"unexpected train size {len(train)}"

def micro_f1(y, p):
    tp = (y * p).sum(); fp = ((1 - y) * p).sum(); fn = (y * (1 - p)).sum()
    d = 2 * tp + fp + fn
    return 2.0 * tp / d if d > 0 else 0.0

def best_global_threshold(y, scores, grid):
    return float(grid[int(np.argmax([micro_f1(y, (scores >= t).astype(int)) for t in grid]))])

def apply_thresholds(scores, t):
    p = (scores >= t).astype(int)
    empty = p.sum(axis=1) == 0                       # never predict "no technique"
    p[np.where(empty)[0], scores[empty].argmax(axis=1)] = 1
    return p

def evaluate(y, p):
    present = y.sum(axis=0) > 0                      # macro-F1 only over classes present in this split
    return {
        "micro_f1": f1_score(y, p, average="micro", zero_division=0),
        "macro_f1": f1_score(y[:, present], p[:, present], average="macro", zero_division=0),
        "micro_precision": precision_score(y, p, average="micro", zero_division=0),
        "micro_recall": recall_score(y, p, average="micro", zero_division=0),
        "subset_accuracy": float((y == p).all(axis=1).mean()),
        "hamming_loss": hamming_loss(y, p),
        "n_classes_present": int(present.sum()),
    }

def ranking_metrics(y, scores):
    """hit@1 = top guess is a gold label. mrr@3 = 1/rank of first gold label in top 3 (0 if none).
    Rows whose gold labels are all unseen-in-train count as misses."""
    top3 = np.argsort(-scores, axis=1)[:, :3]
    hits = np.take_along_axis(y, top3, axis=1)       # rows x 3 of 0/1
    hit1 = hits[:, 0].mean()
    first = np.where(hits.any(axis=1), hits.argmax(axis=1) + 1, np.inf)
    return {"hit@1": float(hit1), "mrr@3": float(np.mean(1.0 / first))}

def set_seed(seed):
    import torch
    random.seed(seed); np.random.seed(seed)
    torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)

train=3468 dev=604 test=725 classes=180


In [4]:
# Cell 3 - TF-IDF + LinearSVC reference baseline (CPU, seconds)
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.multiclass import OneVsRestClassifier
from sklearn.svm import LinearSVC

vec = TfidfVectorizer(ngram_range=(1, 2), sublinear_tf=True).fit(train["text"])
Xtr, Xdv, Xte = (vec.transform(d["text"]) for d in (train, dev, test))
svm = OneVsRestClassifier(LinearSVC(C=1.0, random_state=42)).fit(Xtr, Y_train)
sc_dv, sc_te = svm.decision_function(Xdv), svm.decision_function(Xte)
t_svm = best_global_threshold(Y_dev, sc_dv, np.arange(-0.6, 0.31, 0.05))   # threshold picked on DEV only
svm_metrics = {**evaluate(Y_test, apply_thresholds(sc_te, t_svm)), **ranking_metrics(Y_test, sc_te)}
print(f"SVM threshold={t_svm:.2f}")
print({k: round(v, 4) for k, v in svm_metrics.items()})

SVM threshold=-0.45
{'micro_f1': 0.6961, 'macro_f1': 0.5427, 'micro_precision': 0.6722, 'micro_recall': 0.7218, 'subset_accuracy': 0.5752, 'hamming_loss': 0.0041, 'n_classes_present': 117, 'hit@1': 0.7421, 'mrr@3': 0.8138}


In [6]:
# Cell 4 - transformer training (one clean train_model)
import torch, torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from tqdm.auto import tqdm
from transformers import AutoTokenizer, AutoModelForSequenceClassification, get_linear_schedule_with_warmup

PROB_GRID = np.arange(0.1, 0.9, 0.05)

class TramDS(Dataset):
    def __init__(self, texts, Y, tok, max_length):
        self.enc = tok(list(texts), truncation=True, max_length=max_length)
        self.Y = Y
    def __len__(self): return len(self.Y)
    def __getitem__(self, i):
        item = {k: v[i] for k, v in self.enc.items()}
        item["labels"] = self.Y[i]
        return item

def make_collate(tok):
    def collate(batch):
        y = torch.tensor(np.stack([b["labels"] for b in batch]), dtype=torch.float)
        enc = tok.pad([{k: v for k, v in b.items() if k != "labels"} for b in batch], return_tensors="pt")
        enc["labels"] = y
        return enc
    return collate

def run_epoch(model, loader, loss_fn, device, optimizer=None, scheduler=None):
    training = optimizer is not None
    model.train() if training else model.eval()
    total, probs, labs = 0.0, [], []
    with torch.set_grad_enabled(training):
        for batch in tqdm(loader, disable=True):          # progress bars off -> clean logs
            batch = {k: v.to(device) for k, v in batch.items()}
            y = batch.pop("labels")
            logits = model(**batch).logits
            loss = loss_fn(logits, y)
            if training:
                optimizer.zero_grad()
                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                optimizer.step(); scheduler.step()
            total += loss.item() * y.size(0)
            probs.append(torch.sigmoid(logits).detach().cpu().numpy())
            labs.append(y.cpu().numpy())
    return total / len(loader.dataset), np.concatenate(probs), np.concatenate(labs)

def train_model(model_name, output_dir, seed=42, epochs=30, patience=5, batch_size=16, lr=2e-5,
                head_lr=1e-3, max_length=128, warmup_ratio=0.1, use_pos_weight=True,
                pos_weight_cap=20.0, save_model=False):
    set_seed(seed)
    os.makedirs(output_dir, exist_ok=True)
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    n_cls = len(mlb.classes_)
    print(f"[{model_name} seed={seed}] device={device} classes={n_cls} epochs<={epochs} "
          f"lr={lr} head_lr={head_lr} pos_weight={use_pos_weight}")

    tok = AutoTokenizer.from_pretrained(model_name)
    col = make_collate(tok)
    tr_loader = DataLoader(TramDS(train["text"], Y_train, tok, max_length), batch_size=batch_size, shuffle=True, collate_fn=col)
    dv_loader = DataLoader(TramDS(dev["text"], Y_dev, tok, max_length), batch_size=batch_size * 2, collate_fn=col)
    te_loader = DataLoader(TramDS(test["text"], Y_test, tok, max_length), batch_size=batch_size * 2, collate_fn=col)

    model = AutoModelForSequenceClassification.from_pretrained(
        model_name, num_labels=n_cls, problem_type="multi_label_classification").to(device)

    pw = None
    if use_pos_weight:
        pos = np.clip(Y_train.sum(axis=0), 1, None)
        pw = torch.tensor(np.clip((len(Y_train) - pos) / pos, None, pos_weight_cap), dtype=torch.float, device=device)
    loss_fn = nn.BCEWithLogitsLoss(pos_weight=pw)

    head = [p for n, p in model.named_parameters() if n.startswith("classifier.")]
    body = [p for n, p in model.named_parameters() if not n.startswith("classifier.")]
    assert len(head) > 0, "no classifier.* parameters found"
    optimizer = torch.optim.AdamW([{"params": body, "lr": lr}, {"params": head, "lr": head_lr if head_lr else lr}])
    total_steps = len(tr_loader) * epochs
    scheduler = get_linear_schedule_with_warmup(optimizer, int(total_steps * warmup_ratio), total_steps)

    best_f1, best_epoch, best_state, bad, history = -1.0, 0, None, 0, []
    for ep in range(1, epochs + 1):
        tr_loss, _, _ = run_epoch(model, tr_loader, loss_fn, device, optimizer, scheduler)
        dv_loss, dv_p, dv_y = run_epoch(model, dv_loader, loss_fn, device)
        m = evaluate(dv_y, apply_thresholds(dv_p, best_global_threshold(dv_y, dv_p, PROB_GRID)))
        history.append({"epoch": ep, "train_loss": tr_loss, "dev_loss": dv_loss, "dev_micro_f1": m["micro_f1"], "dev_macro_f1": m["macro_f1"]})
        print(f"  epoch {ep:2d}: train_loss={tr_loss:.4f} dev_loss={dv_loss:.4f} dev_micro_f1={m['micro_f1']:.4f} dev_macro_f1={m['macro_f1']:.4f}")
        if m["micro_f1"] > best_f1:
            best_f1, best_epoch, bad = m["micro_f1"], ep, 0
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
        else:
            bad += 1
            if bad >= patience:
                print(f"  early stop (no dev improvement for {patience} epochs)")
                break

    model.load_state_dict(best_state)
    _, dv_p, dv_y = run_epoch(model, dv_loader, loss_fn, device)
    thr = best_global_threshold(dv_y, dv_p, PROB_GRID)                 # threshold from DEV
    _, te_p, te_y = run_epoch(model, te_loader, loss_fn, device)
    metrics = {**evaluate(te_y, apply_thresholds(te_p, thr)), **ranking_metrics(te_y, te_p)}
    print("  TEST:", {k: round(v, 4) if isinstance(v, float) else v for k, v in metrics.items()})

    np.save(f"{output_dir}/test_probs.npy", te_p); np.save(f"{output_dir}/dev_probs.npy", dv_p)
    np.save(f"{output_dir}/test_labels.npy", te_y); np.save(f"{output_dir}/dev_labels.npy", dv_y)
    np.save(f"{output_dir}/mlb_classes.npy", mlb.classes_)
    json.dump({"model_name": model_name, "seed": seed, "best_epoch": best_epoch, "dev_best_micro_f1": best_f1,
               "threshold": thr, "test": metrics, "history": history,
               "config": dict(epochs=epochs, patience=patience, batch_size=batch_size, lr=lr, head_lr=head_lr,
                              max_length=max_length, use_pos_weight=use_pos_weight, pos_weight_cap=pos_weight_cap)},
              open(f"{output_dir}/metrics.json", "w"), indent=2)
    if save_model:
        model.save_pretrained(output_dir); tok.save_pretrained(output_dir)
    print(f"  saved to {output_dir} (best epoch {best_epoch})")
    del model, optimizer, best_state
    import gc; gc.collect(); torch.cuda.empty_cache()
    return metrics

In [7]:
# Cell 5 - the 4 runs (skips any run that already has metrics.json, so you can resume after a disconnect)
print("OUT_ROOT =", OUT_ROOT)
assert OUT_ROOT.endswith("run_tram_2_epoc_30"), "OUT_ROOT is wrong - fix before training!"

RUNS = [  # (tag, use_pos_weight)
    ("roberta", False), ("secbert", False),     # no class weights first (most likely to help)
    ("roberta", True),  ("secbert", True),
]
for tag, pw in RUNS:
    out = f"{OUT_ROOT}/{tag}_{'pw' if pw else 'nopw'}_s42"
    if os.path.exists(f"{out}/metrics.json"):
        print("skip", out); continue
    print(f"\n>>> {tag}  use_pos_weight={pw}")
    train_model(MODELS[tag], out, seed=42, epochs=30, patience=5, lr=2e-5, head_lr=1e-3,
                use_pos_weight=pw, pos_weight_cap=20.0)

OUT_ROOT = /content/drive/MyDrive/KDU/Thesis/run_tram_2_epoc_30
skip /content/drive/MyDrive/KDU/Thesis/run_tram_2_epoc_30/roberta_nopw_s42
skip /content/drive/MyDrive/KDU/Thesis/run_tram_2_epoc_30/secbert_nopw_s42
skip /content/drive/MyDrive/KDU/Thesis/run_tram_2_epoc_30/roberta_pw_s42

>>> secbert  use_pos_weight=True
[jackaduma/SecBERT seed=42] device=cuda classes=180 epochs<=30 lr=2e-05 head_lr=0.001 pos_weight=True


config.json:   0%|          | 0.00/467 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/378k [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  336MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: jackaduma/SecBERT
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


  epoch  1: train_loss=0.4676 dev_loss=0.3006 dev_micro_f1=0.1440 dev_macro_f1=0.0170
  epoch  2: train_loss=0.2485 dev_loss=0.1916 dev_micro_f1=0.4083 dev_macro_f1=0.1811
  epoch  3: train_loss=0.1419 dev_loss=0.1270 dev_micro_f1=0.5415 dev_macro_f1=0.3211
  epoch  4: train_loss=0.0802 dev_loss=0.1085 dev_micro_f1=0.5986 dev_macro_f1=0.4116
  epoch  5: train_loss=0.0474 dev_loss=0.1024 dev_micro_f1=0.6070 dev_macro_f1=0.4196
  epoch  6: train_loss=0.0312 dev_loss=0.1162 dev_micro_f1=0.6546 dev_macro_f1=0.4641
  epoch  7: train_loss=0.0228 dev_loss=0.1167 dev_micro_f1=0.6474 dev_macro_f1=0.4869
  epoch  8: train_loss=0.0168 dev_loss=0.1217 dev_micro_f1=0.6643 dev_macro_f1=0.4895
  epoch  9: train_loss=0.0127 dev_loss=0.1348 dev_micro_f1=0.6849 dev_macro_f1=0.5100
  epoch 10: train_loss=0.0104 dev_loss=0.1382 dev_micro_f1=0.6882 dev_macro_f1=0.5325
  epoch 11: train_loss=0.0082 dev_loss=0.1522 dev_micro_f1=0.6802 dev_macro_f1=0.5067
  epoch 12: train_loss=0.0067 dev_loss=0.1501 dev_micr

In [8]:
# Cell 6 - results table (choose pw vs nopw by DEV, never by test)
rows = []
for tag in MODELS:
    for pw in ("pw", "nopw"):
        p = f"{OUT_ROOT}/{tag}_{pw}_s42/metrics.json"
        if os.path.exists(p):
            r = json.load(open(p)); t = r["test"]
            rows.append({"run": f"{tag}_{pw}", "best_epoch": r["best_epoch"], "dev_micro_f1": round(r["dev_best_micro_f1"], 4),
                         "test_micro_f1": round(t["micro_f1"], 4), "test_macro_f1": round(t["macro_f1"], 4),
                         "precision": round(t["micro_precision"], 4), "recall": round(t["micro_recall"], 4),
                         "hit@1": round(t["hit@1"], 4), "mrr@3": round(t["mrr@3"], 4)})
rows.append({"run": "TF-IDF+SVM (reference)", "best_epoch": "-", "dev_micro_f1": "-",
             "test_micro_f1": round(svm_metrics["micro_f1"], 4), "test_macro_f1": round(svm_metrics["macro_f1"], 4),
             "precision": round(svm_metrics["micro_precision"], 4), "recall": round(svm_metrics["micro_recall"], 4),
             "hit@1": round(svm_metrics["hit@1"], 4), "mrr@3": round(svm_metrics["mrr@3"], 4)})
res = pd.DataFrame(rows)
print(res.to_string(index=False))
for tag in MODELS:
    sub = res[res["run"].isin([f"{tag}_pw", f"{tag}_nopw"])]
    if len(sub) == 2:
        print(f"\n{tag}: better on DEV -> {sub.loc[sub['dev_micro_f1'].astype(float).idxmax(), 'run']}")
res

                   run best_epoch dev_micro_f1  test_micro_f1  test_macro_f1  precision  recall  hit@1  mrr@3
            secbert_pw         18       0.6993         0.7042         0.5363     0.7245  0.6849 0.7434 0.8055
          secbert_nopw         16        0.697         0.6888         0.5173     0.7119  0.6671 0.7338 0.7977
            roberta_pw         25        0.736         0.7077         0.5681     0.7268  0.6897 0.7572 0.8113
          roberta_nopw         19        0.736         0.7124         0.5810     0.7423  0.6849 0.7614 0.8138
TF-IDF+SVM (reference)          -            -         0.6961         0.5427     0.6722  0.7218 0.7421 0.8138

secbert: better on DEV -> secbert_pw

roberta: better on DEV -> roberta_pw


,run,best_epoch,dev_micro_f1,test_micro_f1,test_macro_f1,precision,recall,hit@1,mrr@3
0,secbert_pw,18,0.6993,0.7042,0.5363,0.7245,0.6849,0.7434,0.8055
1,secbert_nopw,16,0.697,0.6888,0.5173,0.7119,0.6671,0.7338,0.7977
2,roberta_pw,25,0.736,0.7077,0.5681,0.7268,0.6897,0.7572,0.8113
3,roberta_nopw,19,0.736,0.7124,0.5810,0.7423,0.6849,0.7614,0.8138
4,TF-IDF+SVM (reference),-,-,0.6961,0.5427,0.6722,0.7218,0.7421,0.8138
